In [ ]:
# MFMC Demo for Mahnob-HCI Dataset - Subject-Dependent One-Fold Smoke Test
# Multi-modal Feature Matching and Correlation (MFMC) approach
# Modalities: EEG, ECG, Temperature signals

import os
import time
import pickle
from datetime import timedelta

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("MFMC Fusion MLP Demo for Mahnob-HCI Dataset")
print("=" * 60)

def select_device(min_free_memory_mb=10000):
    """Select the CUDA GPU with the most free VRAM, or stop if none is suitable."""
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is available. Stop this notebook instead of running on CPU.")

    try:
        import subprocess
        query = [
            "nvidia-smi",
            "--query-gpu=index,memory.free,memory.total,memory.used",
            "--format=csv,noheader,nounits",
        ]
        output = subprocess.check_output(query, encoding="utf-8")
        gpu_stats = []
        for line in output.strip().splitlines():
            index, free_mem, total_mem, used_mem = [part.strip() for part in line.split(",")]
            gpu_stats.append({
                "index": int(index),
                "free": int(free_mem),
                "total": int(total_mem),
                "used": int(used_mem),
            })
        best_gpu = max(gpu_stats, key=lambda item: item["free"])
        if best_gpu["free"] < min_free_memory_mb:
            raise RuntimeError(
                f"Best GPU has only {best_gpu['free']} MB free; require at least {min_free_memory_mb} MB."
            )
        torch.cuda.set_device(best_gpu["index"])
        device = torch.device(f"cuda:{best_gpu['index']}")
        print(
            f"Selected GPU {best_gpu['index']} "
            f"({best_gpu['free']} MB free / {best_gpu['total']} MB total)."
        )
        return device
    except Exception as exc:
        print(f"GPU auto-selection failed ({exc}); using cuda:0.")
        return torch.device("cuda:0")


In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

PROJECT_ROOT = os.environ.get('TAFFC_MFMC_ROOT', '/home/zhengdeyang/TAFFC_MFMC')
BASE_PATH = f'{PROJECT_ROOT}/MFMC/HCI'
DATA_DIR = os.environ.get('HCI_PROCESSED_DIR', f'{BASE_PATH}/Data_processed')
RESULTS_DIR = f'{BASE_PATH}/MFMC_Fusion_MLP/results/subject_dep_one_fold_test'

BATCH_SIZE = 100
TOTAL_ITERATIONS = 1001
EVAL_INTERVAL = 500
N_FOLDS = 5
RANDOM_SEED = 42

LEARNING_RATE_ENCODER = 0.0001
LEARNING_RATE_CLASSIFIER = 0.0001
COV_BETA = 0.5
USE_CLASS_BALANCING = False
FEATURE_DIM = 128
NUM_CLASSES = 4
QUADRANT_NAMES = ['LVLA', 'LVHA', 'HVLA', 'HVHA']

os.makedirs(RESULTS_DIR, exist_ok=True)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Configuration loaded successfully!")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Total iterations per fold: {TOTAL_ITERATIONS}")
print(f"Batch size: {BATCH_SIZE}")


In [ ]:
# =============================================================================
# MFMC LOSS FUNCTIONS AND PROJECTION HEAD
# =============================================================================

def adaptive_estimation(v_t, beta, square_term, i):
    """Adaptive smoothing filter for estimating covariances."""
    v_t = beta * v_t + (1 - beta) * square_term.detach()
    return v_t, (v_t / (1 - beta ** i))

def mfmc_t_trace(x, y, track_cov, i, cov_beta=0.95):
    """Compute the MFMC-T trace loss for one feature/projection pair."""
    Rx = (x.T @ x) / x.shape[0]
    Ry = (y.T @ y) / y.shape[0]
    Pxy = (x.T @ y) / x.shape[0]

    eps = 1e-6
    Rx = Rx + torch.eye(Rx.shape[0], device=Rx.device) * eps
    Ry = Ry + torch.eye(Ry.shape[0], device=Ry.device) * eps

    track_cov['Rx'], Rx_est = adaptive_estimation(track_cov['Rx'], cov_beta, Rx, i)
    track_cov['Ry'], Ry_est = adaptive_estimation(track_cov['Ry'], cov_beta, Ry, i)
    track_cov['Pxy'], Pxy_est = adaptive_estimation(track_cov['Pxy'], cov_beta, Pxy, i)

    Rx_est_inv = torch.inverse(Rx_est)
    Ry_est_inv = torch.inverse(Ry_est)

    cost = -Rx_est_inv @ Rx @ Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy_est.T \
           + Rx_est_inv @ Pxy @ Ry_est_inv @ Pxy_est.T \
           - Rx_est_inv @ Pxy_est @ Ry_est_inv @ Ry @ Ry_est_inv @ Pxy_est.T \
           + Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy.T
    return track_cov, -torch.trace(cost)

def tri_modal_projection_loss(fe1, fe2, fe3, proj12, proj23, proj13, trackers, step, cov_beta=0.5):
    """MFMC tri-modal projection loss: each modality matches a projection of the other two."""
    concat_12 = torch.cat([fe1, fe2], dim=1)
    concat_23 = torch.cat([fe2, fe3], dim=1)
    concat_13 = torch.cat([fe1, fe3], dim=1)

    p_12 = proj12(concat_12)
    p_23 = proj23(concat_23)
    p_13 = proj13(concat_13)

    trackers['track_1_23'], loss1 = mfmc_t_trace(fe1, p_23, trackers['track_1_23'], step, cov_beta)
    trackers['track_2_13'], loss2 = mfmc_t_trace(fe2, p_13, trackers['track_2_13'], step, cov_beta)
    trackers['track_3_12'], loss3 = mfmc_t_trace(fe3, p_12, trackers['track_3_12'], step, cov_beta)
    return trackers, loss1 + loss2 + loss3

class ProjectionHead(nn.Module):
    """Two-layer MLP projection head for pairwise modality fusion."""
    def __init__(self, input_dim=256, hidden_dim=512, output_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        return self.bn2(self.fc2(x))


In [ ]:
# =============================================================================
# NEURAL NETWORK ARCHITECTURES
# =============================================================================

class ComplexClassifier(nn.Module):
    def __init__(self, dim_features=128, num_classes=4):
        super().__init__()
        self.fc1 = nn.Linear(dim_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        x = torch.relu(self.bn3(self.fc3(x)))
        return self.fc4(x)

class NETWORK_F_MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=200, out_dim=200, num_layers=2):
        super().__init__()
        self.fc_list = nn.ModuleList()
        self.bn_list = nn.ModuleList()
        self.fc_list.append(nn.Linear(input_dim, hidden_dim, bias=True))
        self.bn_list.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.fc_list.append(nn.Linear(hidden_dim, hidden_dim, bias=True))
            self.bn_list.append(nn.BatchNorm1d(hidden_dim))
        self.fc_final = nn.Linear(hidden_dim, out_dim, bias=True)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)
        for fc, bn in zip(self.fc_list, self.bn_list):
            x = torch.relu(bn(fc(x)))
        return torch.sigmoid(self.fc_final(x))

class Advanced1DCNN_channel(nn.Module):
    def __init__(self, input_channels=1, num_classes=128, input_size=2560):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5), nn.BatchNorm1d(32),
            nn.ReLU(), nn.MaxPool1d(kernel_size=4, stride=4))
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=11, padding=5), nn.BatchNorm1d(64),
            nn.ReLU(), nn.MaxPool1d(kernel_size=4, stride=4))
        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=11, padding=5), nn.BatchNorm1d(128),
            nn.ReLU(), nn.MaxPool1d(kernel_size=4, stride=4))
        self.conv4 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=11, padding=5), nn.BatchNorm1d(256),
            nn.ReLU(), nn.MaxPool1d(kernel_size=4, stride=4))

        with torch.no_grad():
            feat_size = self._get_conv_output_size(input_size)

        self.fc1 = nn.Sequential(nn.Linear(256 * feat_size, 1024), nn.BatchNorm1d(1024), nn.ReLU())
        self.fc2 = nn.Sequential(nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU())
        self.fc3 = nn.Linear(512, num_classes)
        self.MLP = NETWORK_F_MLP(
            input_dim=num_classes * input_channels,
            hidden_dim=4000,
            out_dim=num_classes,
            num_layers=1,
        )

    def _get_conv_output_size(self, length):
        x = torch.zeros(1, 1, length)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        return x.shape[2]

    def forward(self, x):
        bs, channels, _ = x.shape
        x = x.view(bs * channels, 1, -1)
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = self.conv4(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)
        out = out.view(bs, channels, -1)
        out = out.flatten(1)
        return self.MLP(out)


In [ ]:
# =============================================================================
# DATA LOADING AND SUBJECT-DEPENDENT SPLITTING
# =============================================================================

print("Loading Mahnob-HCI dataset...")
print("Modalities: EEG, ECG, Temperature")

try:
    subject = torch.from_numpy(np.load(f'{DATA_DIR}/subject.npy')).long()
    emotion_labels = torch.from_numpy(np.load(f'{DATA_DIR}/emotion_labels.npy')).long()
    eeg_data = torch.from_numpy(np.load(f'{DATA_DIR}/eeg_data.npy')).float()
    ecg_data = torch.from_numpy(np.load(f'{DATA_DIR}/ecg_data.npy')).float()
    temp_data = torch.from_numpy(np.load(f'{DATA_DIR}/temp_data.npy')).float()

    print("\nData loaded successfully!")
    print(f"- Total samples: {eeg_data.shape[0]}")
    print(f"- Subjects: {torch.unique(subject).numel()} -> {torch.unique(subject).tolist()}")
    print(f"- EEG shape: {tuple(eeg_data.shape)}")
    print(f"- ECG shape: {tuple(ecg_data.shape)}")
    print(f"- Temperature shape: {tuple(temp_data.shape)}")
    print(f"- Labels shape: {tuple(emotion_labels.shape)}")
    print(f"- Class distribution: {torch.bincount(emotion_labels).tolist()}")

    data_loaded = True
except FileNotFoundError as exc:
    print(f"Data loading failed: {exc}")
    print("Run MFMC/HCI/HCI_Preprocess.py first or set HCI_PROCESSED_DIR to the processed data directory.")
    data_loaded = False

if data_loaded:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    fold_splits = []
    for fold, (train_indices, test_indices) in enumerate(skf.split(np.zeros(len(emotion_labels)), emotion_labels.numpy()), start=1):
        fold_splits.append({
            'fold': fold,
            'train_indices': torch.as_tensor(train_indices, dtype=torch.long),
            'test_indices': torch.as_tensor(test_indices, dtype=torch.long),
        })
        print(f"Fold {fold}: train={len(train_indices)}, test={len(test_indices)}")


In [ ]:
# =============================================================================
# MODEL INITIALIZATION AND TRAINING HELPERS
# =============================================================================

if data_loaded:
    device = select_device()

    def create_trackers(device, feature_dim=FEATURE_DIM):
        return {
            'track_1_23': {'Rx': torch.zeros(feature_dim, feature_dim, device=device), 'Ry': torch.zeros(feature_dim, feature_dim, device=device), 'Pxy': torch.zeros(feature_dim, feature_dim, device=device)},
            'track_2_13': {'Rx': torch.zeros(feature_dim, feature_dim, device=device), 'Ry': torch.zeros(feature_dim, feature_dim, device=device), 'Pxy': torch.zeros(feature_dim, feature_dim, device=device)},
            'track_3_12': {'Rx': torch.zeros(feature_dim, feature_dim, device=device), 'Ry': torch.zeros(feature_dim, feature_dim, device=device), 'Pxy': torch.zeros(feature_dim, feature_dim, device=device)},
        }

    def create_fold_models(device):
        NET_EEG = Advanced1DCNN_channel(input_channels=eeg_data.shape[1], num_classes=FEATURE_DIM, input_size=eeg_data.shape[2]).to(device)
        NET_ECG = Advanced1DCNN_channel(input_channels=ecg_data.shape[1], num_classes=FEATURE_DIM, input_size=ecg_data.shape[2]).to(device)
        NET_TEMP = Advanced1DCNN_channel(input_channels=temp_data.shape[1], num_classes=FEATURE_DIM, input_size=temp_data.shape[2]).to(device)
        proj_12 = ProjectionHead(input_dim=FEATURE_DIM * 2, hidden_dim=512, output_dim=FEATURE_DIM).to(device)
        proj_23 = ProjectionHead(input_dim=FEATURE_DIM * 2, hidden_dim=512, output_dim=FEATURE_DIM).to(device)
        proj_13 = ProjectionHead(input_dim=FEATURE_DIM * 2, hidden_dim=512, output_dim=FEATURE_DIM).to(device)
        classifier = ComplexClassifier(dim_features=FEATURE_DIM, num_classes=NUM_CLASSES).to(device)
        return NET_EEG, NET_ECG, NET_TEMP, proj_12, proj_23, proj_13, classifier

    def evaluate_model(NET_EEG, classifier, loader):
        NET_EEG.eval()
        classifier.eval()
        correct, total = 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for eeg, ecg, temp, labels in loader:
                eeg, labels = eeg.to(device), labels.to(device)
                outputs = classifier(NET_EEG(eeg))
                predicted = outputs.argmax(dim=1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return correct / total if total else 0.0, np.asarray(all_preds), np.asarray(all_labels)

    def plot_learning_curves(costs, classifier_losses, test_accuracies, save_path, fold=None):
        fold_text = f' (Fold {fold})' if fold is not None else ''
        plt.figure(figsize=(18, 5))
        plt.subplot(1, 3, 1)
        plt.plot(costs)
        plt.title(f'Unsupervised MFMC Cost{fold_text}')
        plt.xlabel('Iteration (x100)')
        plt.grid(True, alpha=0.3)
        plt.subplot(1, 3, 2)
        plt.plot(classifier_losses)
        plt.title(f'Supervised Classifier Loss{fold_text}')
        plt.xlabel('Iteration (x100)')
        plt.grid(True, alpha=0.3)
        plt.subplot(1, 3, 3)
        plt.plot(test_accuracies)
        plt.title(f'Test Accuracy{fold_text}')
        plt.xlabel(f'Evaluation (x{EVAL_INTERVAL} iterations)')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

    def train_one_fold(fold_info):
        fold_num = fold_info['fold']
        fold_dir = os.path.join(RESULTS_DIR, f'fold_{fold_num}')
        os.makedirs(fold_dir, exist_ok=True)

        train_indices = fold_info['train_indices']
        test_indices = fold_info['test_indices']
        train_dataset = TensorDataset(eeg_data[train_indices], ecg_data[train_indices], temp_data[train_indices], emotion_labels[train_indices])
        test_dataset = TensorDataset(eeg_data[test_indices], ecg_data[test_indices], temp_data[test_indices], emotion_labels[test_indices])
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

        if len(train_loader) == 0:
            raise ValueError("The training loader is empty. Reduce BATCH_SIZE or check the split size.")

        NET_EEG, NET_ECG, NET_TEMP, proj_12, proj_23, proj_13, classifier = create_fold_models(device)
        feature_params = list(NET_EEG.parameters()) + list(NET_ECG.parameters()) + list(NET_TEMP.parameters()) + \
                         list(proj_12.parameters()) + list(proj_23.parameters()) + list(proj_13.parameters())
        optimizer_features = optim.Adam(feature_params, lr=LEARNING_RATE_ENCODER, amsgrad=True)
        optimizer_classifier = optim.Adam(classifier.parameters(), lr=LEARNING_RATE_CLASSIFIER, amsgrad=True)

        if USE_CLASS_BALANCING:
            train_labels = emotion_labels[train_indices]
            class_counts = torch.bincount(train_labels, minlength=NUM_CLASSES).float().clamp_min(1.0)
            class_weights = 1.0 / class_counts
            class_weights = class_weights / class_weights.sum() * NUM_CLASSES
            criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
            print(f"Fold {fold_num} class weights: {class_weights.tolist()}")
        else:
            criterion = nn.CrossEntropyLoss()

        trackers = create_trackers(device)
        costs, classifier_losses, test_accuracies = [], [], []
        best_accuracy = 0.0
        best_model_path = os.path.join(fold_dir, f'best_model_fold_{fold_num}.pth')

        start_time = time.time()
        last_eval_time = start_time
        train_iter = iter(train_loader)

        print(f"{'=' * 70}")
        print(f"TRAINING FOLD {fold_num}/{N_FOLDS}: train={len(train_indices)}, test={len(test_indices)}")
        if 'test_subjects' in fold_info:
            print(f"Test subjects: {fold_info['test_subjects'].tolist()}")
        print(f"{'=' * 70}")

        for i in range(1, TOTAL_ITERATIONS):
            try:
                input_eeg, input_ecg, input_temp, labels_batch = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                input_eeg, input_ecg, input_temp, labels_batch = next(train_iter)

            input_eeg = input_eeg.to(device)
            input_ecg = input_ecg.to(device)
            input_temp = input_temp.to(device)
            labels_batch = labels_batch.to(device)

            optimizer_features.zero_grad()
            feature_eeg = NET_EEG(input_eeg)
            feature_ecg = NET_ECG(input_ecg)
            feature_temp = NET_TEMP(input_temp)
            trackers, cost = tri_modal_projection_loss(
                feature_eeg, feature_ecg, feature_temp,
                proj_12, proj_23, proj_13, trackers, i, COV_BETA,
            )
            cost.backward()
            optimizer_features.step()

            optimizer_classifier.zero_grad()
            with torch.no_grad():
                feature_eeg_detached = NET_EEG(input_eeg)
            logits = classifier(feature_eeg_detached.detach())
            cls_loss = criterion(logits, labels_batch)
            cls_loss.backward()
            optimizer_classifier.step()

            if i % 100 == 0:
                costs.append(cost.item())
                classifier_losses.append(cls_loss.item())
                print(f"Iter {i} | MFMC Cost {cost.item():.4f} | Classifier Loss {cls_loss.item():.4f}")

            if i % EVAL_INTERVAL == 0:
                current_time = time.time()
                elapsed = current_time - start_time
                eta = (elapsed / i) * (TOTAL_ITERATIONS - i)
                accuracy, _, _ = evaluate_model(NET_EEG, classifier, test_loader)
                test_accuracies.append(accuracy)
                print(
                    f"** Iter {i} | Test Accuracy: {accuracy:.4f} | "
                    f"Elapsed: {timedelta(seconds=int(elapsed))} | "
                    f"Time/{EVAL_INTERVAL}iters: {current_time - last_eval_time:.1f}s | "
                    f"ETA: {timedelta(seconds=int(eta))} **"
                )
                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    torch.save({
                        'eeg_model': NET_EEG.state_dict(),
                        'ecg_model': NET_ECG.state_dict(),
                        'temp_model': NET_TEMP.state_dict(),
                        'classifier': classifier.state_dict(),
                    }, best_model_path)
                    print(f"New best accuracy found. Saved model to {best_model_path}")
                NET_EEG.train(); NET_ECG.train(); NET_TEMP.train(); classifier.train()
                last_eval_time = current_time

        if not os.path.exists(best_model_path):
            torch.save({
                'eeg_model': NET_EEG.state_dict(),
                'ecg_model': NET_ECG.state_dict(),
                'temp_model': NET_TEMP.state_dict(),
                'classifier': classifier.state_dict(),
            }, best_model_path)

        checkpoint = torch.load(best_model_path, map_location=device)
        NET_EEG.load_state_dict(checkpoint['eeg_model'])
        classifier.load_state_dict(checkpoint['classifier'])
        final_accuracy, all_preds, all_labels = evaluate_model(NET_EEG, classifier, test_loader)

        report = classification_report(all_labels, all_preds, target_names=QUADRANT_NAMES, zero_division=0)
        with open(os.path.join(fold_dir, f'final_report_fold_{fold_num}.txt'), 'w') as f:
            f.write(report)

        cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=QUADRANT_NAMES)
        disp.plot(cmap=plt.cm.Blues)
        plt.title(f'Confusion Matrix - Fold {fold_num}')
        plt.savefig(os.path.join(fold_dir, f'confusion_matrix_fold_{fold_num}.png'), dpi=300, bbox_inches='tight')
        plt.close()

        plot_learning_curves(costs, classifier_losses, test_accuracies, os.path.join(fold_dir, f'learning_curves_fold_{fold_num}.png'), fold_num)

        fold_result = {
            'fold': fold_num,
            'best_accuracy': best_accuracy,
            'final_accuracy': final_accuracy,
            'costs': costs,
            'classifier_losses': classifier_losses,
            'test_accuracies': test_accuracies,
            'all_preds': all_preds,
            'all_labels': all_labels,
            'report': report,
        }
        with open(os.path.join(fold_dir, f'fold_{fold_num}_results.pkl'), 'wb') as f:
            pickle.dump(fold_result, f)
        print(f"Fold {fold_num} finished. Best accuracy: {best_accuracy:.4f}; final accuracy: {final_accuracy:.4f}")
        return fold_result

    fold_results = []


In [ ]:
# =============================================================================
# TRAINING SETUP - PREPARE FOR 5 SEPARATE FOLD TRAINING CELLS
# =============================================================================

if data_loaded:
    print("Preparing 5-fold subject-dependent MFMC training...")
    print("Phase 1: unsupervised encoder training with MFMC tri-modal projection loss")
    print("Phase 2: supervised classifier training with CrossEntropy loss on EEG features")
    print(f"Each fold runs for {TOTAL_ITERATIONS} iterations.")


In [ ]:
# =============================================================================
# ONE-FOLD TRAINING
# =============================================================================

if data_loaded:
    print("Running a single-fold smoke test with fold 1.")
    result = train_one_fold(fold_splits[0])
    fold_results.append(result)


In [ ]:
# =============================================================================
# SINGLE-FOLD EVALUATION SUMMARY
# =============================================================================

if data_loaded and fold_results:
    result = fold_results[0]
    print("\nHCI MFMC Fusion MLP one-fold smoke-test result")
    print("=" * 60)
    print(f"Best accuracy: {result['best_accuracy']:.4f}")
    print(f"Final accuracy: {result['final_accuracy']:.4f}")
    print("\nClassification report:\n")
    print(result['report'])

    with open(os.path.join(RESULTS_DIR, 'single_fold_test_results.pkl'), 'wb') as f:
        pickle.dump(result, f)
else:
    print("No result available yet. Run the one-fold training cell first.")
